# Random Forest Model Training -  Elemental Features

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, ParameterGrid
import matplotlib.pyplot as plt
import pickle
import os
import warnings
import time
import multiprocessing as mp
from collections import defaultdict
warnings.filterwarnings('ignore')

SEP = os.sep
random_state = 21
print('Libraries imported successfully!')


## Load and Prepare Data

In [ ]:
# Load data
print('Loading data...')
df = pd.read_excel('PGM_CMP_01_norm_feature.xlsx').sample(frac=1, random_state=random_state, ignore_index=True)

print('\nFirst 5 rows of data:')
display(df.head())

y_prop = 'Mass_Change'
feat_names = ['Time', 'Ni', 'Al', 'Pt', 'Pd', 'Ir', 'Rh']

X = np.array(df[feat_names])
Y = np.array(df[y_prop])

print(f'\nX shape: {X.shape}')
print(f'Y shape: {Y.shape}')
print(f'\nFeatures: {feat_names}')
print(f'Target: {y_prop}')


## Create Output Directories

In [ ]:
from sklearn.ensemble import RandomForestRegressor

ALGO        = 'random_forest'
MODEL_DIR   = 'trained_mod_CMP_rf'
PARITY_DIR  = 'rf_parity_plots'
RESULTS_DIR = 'rf_results'
USE_SCALER  = False   

for d in [MODEL_DIR, PARITY_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'✓ Created: {d}')


def build_model_from_params(p):
    return RandomForestRegressor(**p, random_state=random_state, n_jobs=-1)


def X_tr_scaled(x): return x


## Define Hyperparameter Grid

In [ ]:
param_grid = {
    'n_estimators':     [100, 300, 500, 800, 1000],
    'max_depth':        [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf':  [1, 2, 4, 8],
    'max_features':      ['sqrt', 'log2', 0.5, 0.8],
    'bootstrap':         [True, False],
    'max_samples':       [None, 0.7, 0.8, 0.9],   # only used when bootstrap=True
}

print('=' * 70)
print('HYPERPARAMETER GRID – RANDOM FOREST')
print('=' * 70)
for k, v in param_grid.items():
    print(f'  {k}: {v}')
total = 1
for v in param_grid.values(): total *= len(v)
print(f'\nTotal combinations: {total:,}')
print('Using GridSearchCV with 5-fold CV')


## GridSearchCV

In [ ]:
print('=' * 70)
print('STARTING GRID SEARCH – RANDOM FOREST')
print('=' * 70)

rf_base = RandomForestRegressor(random_state=random_state, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    verbose=1,
    n_jobs=-1,
    return_train_score=True,
)

total_fits = len(list(ParameterGrid(param_grid))) * 5
print(f'Total fits: {total_fits:,}')

start_time = time.time()
grid_search.fit(X, Y)
search_time = time.time() - start_time

print('\n' + '=' * 70)
print('GRID SEARCH COMPLETED')
print('=' * 70)
print(f'Time: {search_time/60:.2f} minutes')
print(f'Best CV MAE: {-grid_search.best_score_:.6f}')
print('\nBest Parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')


## Extract Top Parameter Sets

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results['mean_test_mae'] = -cv_results['mean_test_score']
cv_results_sorted = cv_results.sort_values('mean_test_mae').head(10)

print('=' * 70)
print('TOP 10 PARAMETER COMBINATIONS')
print('=' * 70)

top_params_df = cv_results_sorted[['params', 'mean_test_mae', 'std_test_score']].reset_index(drop=True)
top_params_df.index = range(1, 11)
display(top_params_df)

top_params_df.to_csv(f'{RESULTS_DIR}/top_10_parameter_sets.csv')
print(f'\nSaved -> {RESULTS_DIR}/top_10_parameter_sets.csv')


## K-Fold Cross-Validation for Top 5 Parameter Sets

In [ ]:
top_n_sets = 5
top_param_sets = cv_results_sorted.head(top_n_sets)['params'].tolist()
all_param_set_results = []
Kfold_val = 5

for set_idx, param_set in enumerate(top_param_sets, 1):
    print(f'\n{"="*70}')
    print(f'PARAMETER SET {set_idx}/{top_n_sets}')
    print(f'{"="*70}')
    for k, v in param_set.items(): print(f'  {k}: {v}')

    kf = KFold(n_splits=Kfold_val, shuffle=True, random_state=random_state)
    metrics = {k: [] for k in [
        'train_mae','test_mae','train_mse','test_mse',
        'train_rmse','test_rmse','train_mape','test_mape','train_r2','test_r2']}
    all_predictions = {'train': [], 'test': [], 'y_train': [], 'y_test': []}
    set_start = time.time()
    k_count = 1

    for tr_idx, te_idx in kf.split(X):
        x_tr, x_te = X[tr_idx], X[te_idx]
        y_tr, y_te = Y[tr_idx], Y[te_idx]

        fold_start = time.time()
        mod = RandomForestRegressor(**param_set, random_state=random_state, n_jobs=-1)
        mod.fit(x_tr, y_tr)
        yp_tr = mod.predict(x_tr)
        yp_te = mod.predict(x_te)
        fold_time = time.time() - fold_start

        all_predictions['train'].extend(yp_tr); all_predictions['y_train'].extend(y_tr)
        all_predictions['test'].extend(yp_te);  all_predictions['y_test'].extend(y_te)

        tr_mse = mean_squared_error(y_tr, yp_tr)
        te_mse = mean_squared_error(y_te, yp_te)
        metrics['train_mae'].append(mean_absolute_error(y_tr, yp_tr))
        metrics['test_mae'].append(mean_absolute_error(y_te, yp_te))
        metrics['train_mse'].append(tr_mse); metrics['test_mse'].append(te_mse)
        metrics['train_rmse'].append(np.sqrt(tr_mse)); metrics['test_rmse'].append(np.sqrt(te_mse))
        metrics['train_mape'].append(mean_absolute_percentage_error(y_tr, yp_tr))
        metrics['test_mape'].append(mean_absolute_percentage_error(y_te, yp_te))
        metrics['train_r2'].append(r2_score(y_tr, yp_tr))
        metrics['test_r2'].append(r2_score(y_te, yp_te))

        pickle.dump(mod, open(f'{MODEL_DIR}/rf-mod-set{set_idx:02d}-K{k_count}.pkl', 'wb'))
        print(f'  Fold {k_count}  MAE={metrics["test_mae"][-1]:.6f}  R²={metrics["test_r2"][-1]:.4f}  ({fold_time:.2f}s)')
        k_count += 1

    print(f'\n  Completed in {time.time()-set_start:.2f}s')
    set_summary = {'param_set': set_idx}
    for m in ['mae','mse','rmse','mape','r2']:
        set_summary[f'train_{m}_mean'] = np.mean(metrics[f'train_{m}'])
        set_summary[f'train_{m}_std']  = np.std(metrics[f'train_{m}'])
        set_summary[f'test_{m}_mean']  = np.mean(metrics[f'test_{m}'])
        set_summary[f'test_{m}_std']   = np.std(metrics[f'test_{m}'])
        print(f'  {m.upper()}: Train={np.mean(metrics[f"train_{m}"]):.6f}  Test={np.mean(metrics[f"test_{m}"]):.6f}')

    all_param_set_results.append(set_summary)
    pd.DataFrame(metrics, index=[f'Fold_{i+1}' for i in range(Kfold_val)]).to_csv(
        f'{RESULTS_DIR}/cv_results_set{set_idx:02d}.csv')

    fi_df = pd.DataFrame({'Feature': feat_names, 'Importance': mod.feature_importances_})
    fi_df.sort_values('Importance', ascending=False).to_csv(
        f'{RESULTS_DIR}/feature_importance_set{set_idx:02d}.csv', index=False)

    locals()[f'metrics_set{set_idx}']      = metrics
    locals()[f'predictions_set{set_idx}']  = all_predictions
    locals()[f'params_set{set_idx}']       = param_set

print('\n' + '=' * 70)
print('K-FOLD CV COMPLETED')
print('=' * 70)


## Parity Plots

In [ ]:
print('=' * 70)
print('GENERATING PARITY PLOTS')
print('=' * 70)

for set_idx in range(1, top_n_sets + 1):
    metrics        = locals()[f'metrics_set{set_idx}']
    all_predictions = locals()[f'predictions_set{set_idx}']
    param_set      = locals()[f'params_set{set_idx}']

    fig, ax = plt.subplots(figsize=(10, 10))
    tr_r2  = np.mean(metrics['train_r2'])
    te_r2  = np.mean(metrics['test_r2'])
    tr_mae = np.mean(metrics['train_mae'])
    te_mae = np.mean(metrics['test_mae'])

    ax.scatter(all_predictions['y_train'], all_predictions['train'],
               alpha=0.5, s=50, edgecolors='k', lw=0.5,
               label=f'Train (R²={tr_r2:.3f}, MAE={tr_mae:.4f})')
    ax.scatter(all_predictions['y_test'],  all_predictions['test'],
               alpha=0.5, s=50, edgecolors='k', lw=0.5,
               label=f'Test  (R²={te_r2:.3f}, MAE={te_mae:.4f})')

    y_min, y_max = min(Y), max(Y)
    ax.plot([y_min, y_max], [y_min, y_max], 'k--', lw=2.5, label='Perfect Prediction', alpha=0.7)
    ax.set_xlabel('Experimental Mass Change', fontsize=14, fontweight='bold')
    ax.set_ylabel('Predicted Mass Change',    fontsize=14, fontweight='bold')
    ax.set_title(f'Parity Plot – RF Set {set_idx}\n'
                 f'n_est={param_set["n_estimators"]}, max_depth={param_set["max_depth"]}',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, loc='upper left', framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_aspect('equal', adjustable='box')
    plt.tight_layout()
    plt.savefig(f'{PARITY_DIR}/parity_plot_set{set_idx:02d}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {PARITY_DIR}/parity_plot_set{set_idx:02d}.png')


## Comparison of Parameter Sets

In [ ]:
comparison_df = pd.DataFrame(all_param_set_results).round(6)
print('Test Performance Comparison:')
display(comparison_df[['param_set','test_mae_mean','test_rmse_mean','test_r2_mean']])
comparison_df.to_csv(f'{RESULTS_DIR}/all_parameter_sets_comparison.csv', index=False)
print(f'\nSaved -> {RESULTS_DIR}/all_parameter_sets_comparison.csv')


In [ ]:
# Bar comparison chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics_to_compare = [
    ('test_mae_mean',  'Test MAE',  'lower'),
    ('test_rmse_mean', 'Test RMSE', 'lower'),
    ('test_r2_mean',   'Test R²',   'higher')
]
for idx, (metric, title, better) in enumerate(metrics_to_compare):
    x_pos = np.arange(len(comparison_df))
    bars  = axes[idx].bar(x_pos, comparison_df[metric], alpha=0.8, edgecolor='black', linewidth=1.5)
    best_idx = comparison_df[metric].idxmin() if better == 'lower' else comparison_df[metric].idxmax()
    bars[best_idx].set_color('gold'); bars[best_idx].set_edgecolor('darkred'); bars[best_idx].set_linewidth(2.5)
    for bar in bars:
        h = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., h, f'{h:.4f}',
                       ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Parameter Set', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(title, fontsize=12, fontweight='bold')
    axes[idx].set_title(f'{title} Comparison\n(Gold = Best)', fontsize=13, fontweight='bold')
    axes[idx].set_xticks(x_pos)
    axes[idx].set_xticklabels([f'Set {i+1}' for i in range(len(comparison_df))])
    axes[idx].grid(True, alpha=0.3, axis='y', linestyle='--')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/parameter_sets_comparison.png', dpi=300, bbox_inches='tight'); plt.show()
print(f'Saved -> {RESULTS_DIR}/parameter_sets_comparison.png')


## Per-Composition Consistency Analysis

In [ ]:
# Helper so per-comp cell is self-contained
def build_model_from_params(p):
    return RandomForestRegressor(**p, random_state=random_state, n_jobs=-1)

scaler = None  # not used for RF
USE_SCALER = False
print('=' * 70)
print('PER-COMPOSITION CONSISTENCY ANALYSIS')
print('=' * 70)

best_set_idx   = comparison_df['test_mae_mean'].idxmin() + 1
best_param_set = top_param_sets[best_set_idx - 1]
print(f'Using best param set: Set {best_set_idx}')

comp_records = defaultdict(list)

kf_c = KFold(n_splits=Kfold_val, shuffle=True, random_state=random_state)
print(f'\nTracking predictions across {Kfold_val} folds ...')

for fold_i, (tr_idx, te_idx) in enumerate(kf_c.split(X), 1):
    mod_c = build_model_from_params(best_param_set)
    X_tr_f, X_te_f = (scaler.fit_transform(X[tr_idx]), scaler.transform(X[te_idx])) if USE_SCALER else (X[tr_idx], X[te_idx])
    mod_c.fit(X_tr_f, Y[tr_idx])
    y_pred_te = mod_c.predict(X_te_f)
    for name, actual, pred in zip(df.loc[te_idx, 'alloy_name'].values, Y[te_idx], y_pred_te):
        comp_records[name].append((float(actual), float(pred)))
    mae_f = mean_absolute_error(Y[te_idx], y_pred_te)
    r2_f  = r2_score(Y[te_idx], y_pred_te)
    print(f'  Fold {fold_i}: MAE={mae_f:.5f}  R2={r2_f:.4f}  ({len(te_idx)} rows)')

comp_stats = []
for name, recs in comp_records.items():
    actuals = np.array([r[0] for r in recs])
    preds   = np.array([r[1] for r in recs])
    n_obs   = len(recs)
    mae_c   = float(np.mean(np.abs(actuals - preds)))
    bias_c  = float(np.mean(preds - actuals))
    std_c   = float(np.std(preds)) if n_obs > 1 else 0.0
    r2_c    = float(r2_score(actuals, preds)) if n_obs > 1 else float('nan')
    comp_stats.append({'alloy': name, 'n_test_obs': n_obs,
                       'mean_actual': round(float(actuals.mean()), 5),
                       'mean_pred':   round(float(preds.mean()),   5),
                       'MAE':         round(mae_c,  5),
                       'bias':        round(bias_c, 5),
                       'pred_std':    round(std_c,  5),
                       'R2':          round(r2_c,   4) if r2_c == r2_c else float('nan')})

comp_df = pd.DataFrame(comp_stats).sort_values('MAE').reset_index(drop=True)
comp_df.to_csv(f'{RESULTS_DIR}/per_composition_stats.csv', index=False)
print(f'\nPer-composition table saved -> {RESULTS_DIR}/per_composition_stats.csv')
display(comp_df.head(10))
display(comp_df.tail(10))

# Parity plot per composition
n_comp_plot = len(comp_records)
ncols = 4; nrows = (n_comp_plot + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = axes.flatten()
for ax_i, (comp, recs) in enumerate(sorted(comp_records.items())):
    ax = axes[ax_i]
    actuals = np.array([r[0] for r in recs])
    preds   = np.array([r[1] for r in recs])
    r2_c    = r2_score(actuals, preds) if len(recs) > 1 else float('nan')
    mae_c   = np.mean(np.abs(actuals - preds))
    ax.scatter(actuals, preds, alpha=0.5, s=20, edgecolors='k', lw=0.3)
    mn, mx = min(actuals.min(), preds.min()), max(actuals.max(), preds.max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5)
    ax.set_title(f'{comp}\nR²={r2_c:.3f}  MAE={mae_c:.4f}', fontsize=8)
    ax.set_xlabel('Actual', fontsize=8); ax.set_ylabel('Predicted', fontsize=8)
    ax.grid(True, alpha=0.3)
for ax in axes[n_comp_plot:]:
    ax.set_visible(False)
plt.suptitle('Per-Composition Parity Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_comp_parity_grid.png', dpi=120, bbox_inches='tight'); plt.show()
print(f'Saved -> {RESULTS_DIR}/per_comp_parity_grid.png')


In [ ]:
# -- representative subset: best / middle / worst -------------------------
N_SHOW = 24   
n3     = N_SHOW // 3

rep_df = pd.concat([
    comp_df.head(n3),
    comp_df.iloc[len(comp_df)//2 - n3//2 : len(comp_df)//2 + n3//2],
    comp_df.tail(n3)
]).drop_duplicates('alloy').reset_index(drop=True)

q33 = comp_df['MAE'].quantile(0.33)
q67 = comp_df['MAE'].quantile(0.67)

barcol = [
    '#2ecc71' if v <= q33 else '#e74c3c' if v >= q67 else '#3498db'
    for v in rep_df['MAE']
]

xpos = np.arange(len(rep_df))

# Figure 1 - actual vs predicted with error bars --------------------------
fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(
    xpos,
    rep_df['mean_actual'],
    marker='D',
    s=2,
    c='black',
    zorder=6,
    label='Actual'
)

ax.errorbar(
    xpos,
    rep_df['mean_pred'],
    
    fmt='o',
    ms=7,
    capsize=4,
    elinewidth=1.4,
    color='steelblue',
    ecolor='grey',
    label='Predicted (mean ± std across folds)'
)

ax.set_xticks(xpos)
ax.set_xticklabels(rep_df['alloy'], rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Mass Change', fontsize=12)
ax.set_title(
    f"Per-Composition Consistency – {len(rep_df)} Representative Alloys\n"
    "(green=good | blue=mid | red=challenging)",
    fontsize=12
)

ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(
    'rf_results/comp_actual_vs_pred.png',
    dpi=150,
    bbox_inches='tight'
)
plt.show()
print("Saved -> rf_results/comp_actual_vs_pred.png")

# Figure 2 - MAE bar + bias bar -------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# MAE Plot
axes[0].bar(
    xpos,
    rep_df['MAE'],
    color=barcol,
    edgecolor='k',
    linewidth=0.5
)
axes[0].set_xticks(xpos)
axes[0].set_xticklabels(rep_df['alloy'], rotation=55, ha='right', fontsize=8)
axes[0].set_ylabel('MAE', fontsize=12)
axes[0].set_title(
    "Per-Composition MAE\n"
    "(green=good | blue=mid | red=hard)",
    fontsize=11
)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Bias Plot
biascol = [
    '#e74c3c' if v > 0 else '#3498db'
    for v in rep_df['bias']
]

axes[1].bar(
    xpos,
    rep_df['bias'],
    color=biascol,
    edgecolor='k',
    linewidth=0.5
)
axes[1].axhline(0, color='k', lw=1.2, ls='--')
axes[1].set_xticks(xpos)
axes[1].set_xticklabels(rep_df['alloy'], rotation=55, ha='right', fontsize=8)
axes[1].set_ylabel('Prediction Bias (pred − actual)', fontsize=12)
axes[1].set_title(
    "Per-Composition Bias\n"
    "(red=over-predicted | blue=under-predicted)",
    fontsize=11
)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle(
    'Per-Composition Error & Bias Analysis',
    fontsize=13,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    'rf_results/comp_mae_bias.png',
    dpi=150,
    bbox_inches='tight'
)
plt.show()
print("Saved -> rf_results/comp_mae_bias.png")

# Figure 3 - consistency map (pred_std vs MAE) ----------------------------
fig, ax = plt.subplots(figsize=(9, 7))

sc = ax.scatter(
    comp_df['pred_std'],
    comp_df['MAE'],
    c=comp_df['mean_actual'],
    cmap='viridis',
    s=55,
    alpha=0.8,
    edgecolors='k',
    linewidths=0.3
)

plt.colorbar(sc, ax=ax, label='Mean Actual Mass Change')

for _, row in rep_df.iterrows():
    ax.annotate(
        row['alloy'],
        (row['pred_std'], row['MAE']),
        fontsize=6,
        alpha=0.85,
        xytext=(3, 2),
        textcoords='offset points'
    )

ax.set_xlabel('Prediction Std Dev (spread across folds)', fontsize=12)
ax.set_ylabel('MAE per Composition', fontsize=12)
ax.set_title(
    "Prediction Consistency Map\n"
    "(bottom-left = consistent & accurate | top-right = inconsistent & inaccurate)",
    fontsize=11
)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    'rf_results/comp_consistency_map.png',
    dpi=150,
    bbox_inches='tight'
)
plt.show()
print("Saved -> rf_results/comp_consistency_map.png")

# Figure 4 - R2 histogram over compositions with >1 test obs ---------------
comp_r2 = comp_df.dropna(subset=['R2'])

if len(comp_r2) > 2:
    mean_r2   = comp_r2['R2'].mean()
    median_r2 = comp_r2['R2'].median()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(
        comp_r2['R2'],
        bins=20,
        color='steelblue',
        edgecolor='k',
        alpha=0.8
    )

    ax.axvline(
        mean_r2,
        color='red',
        lw=2,
        ls='--',
        label=f"Mean R2 = {mean_r2:.3f}"
    )
    ax.axvline(
        median_r2,
        color='orange',
        lw=2,
        ls='--',
        label=f"Median R2 = {median_r2:.3f}"
    )

    ax.set_xlabel('R2 per Composition', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(
        "R2 Distribution Across Compositions\n"
        "(compositions with >1 test observation)",
        fontsize=12
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        'rf_results/comp_r2_distribution.png',
        dpi=150,
        bbox_inches='tight'
    )
    plt.show()
    print("Saved -> rf_results/comp_r2_distribution.png")
else:
    print(
        "Note: each composition appears only once in the test set "
        "- per-composition R2 requires >1 test observation."
    )

print("\n[Cell 14 complete]")

## Composition-Aware Learning Curve

In [ ]:
# build_best_model uses best params
def build_best_model():
    return RandomForestRegressor(**best_param_set, random_state=random_state, n_jobs=-1)

USE_SCALER = False
def X_tr_scaled(x): return x
print('=' * 70)
print('COMPOSITION-AWARE LEARNING CURVE')
print('=' * 70)

TEST_FRAC = 0.20
COMP_STEP = 1

assert 'alloy_name' in df.columns, "'alloy_name' column not found in df!"

df_idx = np.arange(len(df))
train_pool_idx, test_idx = train_test_split(df_idx, test_size=TEST_FRAC, random_state=random_state)

X_test_lc = X[test_idx]
Y_test_lc = Y[test_idx]

pool_df = df.iloc[train_pool_idx].copy()
pool_df['_orig_idx'] = train_pool_idx

comp_counts   = pool_df['alloy_name'].value_counts()
all_comps     = comp_counts.index.tolist()
n_comps_total = len(all_comps)

steps = list(range(COMP_STEP, n_comps_total, COMP_STEP))
if steps[-1] != n_comps_total:
    steps.append(n_comps_total)

lc_rows = []

for n_comp_step in steps:
    chosen_comps  = all_comps[:n_comp_step]
    mask          = pool_df['alloy_name'].isin(chosen_comps)
    step_orig_idx = pool_df.loc[mask, '_orig_idx'].values

    X_tr = X[step_orig_idx]
    Y_tr = Y[step_orig_idx]

    model_lc = build_best_model()
    model_lc.fit(X_tr_scaled(X_tr) if USE_SCALER else X_tr, Y_tr)

    yp_test  = model_lc.predict(X_tr_scaled(X_test_lc) if USE_SCALER else X_test_lc)
    yp_train = model_lc.predict(X_tr_scaled(X_tr) if USE_SCALER else X_tr)

    r2_t  = r2_score(Y_test_lc, yp_test)
    mae_t = mean_absolute_error(Y_test_lc, yp_test)
    r2_tr = r2_score(Y_tr, yp_train)
    mae_tr= mean_absolute_error(Y_tr, yp_train)

    lc_rows.append({'n_compositions': n_comp_step, 'n_train_rows': len(X_tr),
                    'r2_test': round(r2_t,5), 'mae_test': round(mae_t,6),
                    'r2_train': round(r2_tr,5), 'mae_train': round(mae_tr,6),
                    'comps_used': ', '.join(chosen_comps)})

    print(f'  {n_comp_step:3d} comps | {len(X_tr):4d} rows | '
          f'R2_test={r2_t:.4f}  MAE_test={mae_t:.5f} | '
          f'R2_train={r2_tr:.4f}  MAE_train={mae_tr:.5f}')

lc_df = pd.DataFrame(lc_rows)
lc_df.to_csv(f'{RESULTS_DIR}/composition_learning_curve.csv', index=False)
print(f'\nSaved -> {RESULTS_DIR}/composition_learning_curve.csv')

full_lc_model   = model_lc
full_train_X    = X[train_pool_idx]
full_train_Y    = Y[train_pool_idx]
full_train_alloys = df.iloc[train_pool_idx]['alloy_name'].values


In [ ]:
ref_row = comparison_df[comparison_df['param_set'] == best_set_idx].iloc[0]
ref_r2  = ref_row['test_r2_mean']
ref_mae = ref_row['test_mae_mean']

nc  = lc_df['n_compositions'].values
ntr = lc_df['n_train_rows'].values
r2_test   = lc_df['r2_test'].values
mae_test  = lc_df['mae_test'].values
r2_train  = lc_df['r2_train'].values
mae_train = lc_df['mae_train'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, y_te, y_tr, ref, ylabel, title in [
        (axes[0], r2_test,  r2_train,  ref_r2,  'R²',  'Composition Learning Curve – R²'),
        (axes[1], mae_test, mae_train, ref_mae, 'MAE', 'Composition Learning Curve – MAE')]:
    ax.plot(nc, y_te, 'o-', color='tomato',    lw=2.2, ms=7, label='Test set')
    ax.plot(nc, y_tr, 's--', color='steelblue', lw=2.2, ms=7, label='Training set')
    ax.axhline(ref, color='darkgreen', lw=1.6, ls='-.', label=f'Full-CV = {ref:.4f}')
    ax.set_xlabel('Number of Compositions in Training Set', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(nc); ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(nc)
    ax2.set_xticklabels(ntr, rotation=45, fontsize=7)
    ax2.set_xlabel('Number of Training Rows', fontsize=9, color='grey')
plt.suptitle(f'Composition-Aware Learning Curves  |  Best Param Set {best_set_idx}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comp_learning_curve_r2_mae.png', dpi=150, bbox_inches='tight'); plt.show()

gap_r2  = r2_train - r2_test
gap_mae = mae_test  - mae_train
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, gap, ylabel, title, col in [
        (axes[0], gap_r2,  'R² Gap (Train − Test)',  'Overfitting Diagnostic – R²',  'darkorchid'),
        (axes[1], gap_mae, 'MAE Gap (Test − Train)', 'Generalisation Gap – MAE',     'chocolate')]:
    ax.plot(nc, gap, 'o-', color=col, lw=2, ms=6)
    ax.fill_between(nc, 0, gap, where=gap > 0, alpha=0.25, color=col, label='Gap region')
    ax.axhline(0, color='k', lw=0.9, ls='--')
    ax.set_xlabel('Number of Compositions', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12); ax.set_title(title, fontsize=12)
    ax.set_xticks(nc); ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('Train–Test Gap: Overfitting / Generalisation Diagnostic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comp_learning_curve_gap.png', dpi=150, bbox_inches='tight'); plt.show()

d_r2  = np.diff(r2_test)
d_mae = np.diff(mae_test)
nc_mid= (nc[:-1] + nc[1:]) / 2.0
bw    = (nc[1] - nc[0]) * 0.6 if len(nc) > 1 else 2.0
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, d_vals, ylabel, title, better in [
        (axes[0], d_r2,  'ΔR² per step',  'Marginal R² Gain',       'max'),
        (axes[1], d_mae, 'ΔMAE per step', 'Marginal MAE Reduction', 'min')]:
    colors = ['#27ae60' if (v > 0 if better=='max' else v < 0) else '#e74c3c' for v in d_vals]
    ax.bar(nc_mid, d_vals, width=bw, color=colors, edgecolor='k', linewidth=0.4)
    ax.axhline(0, color='k', lw=0.9)
    ax.set_xlabel('Midpoint Composition Count', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11); ax.set_title(title, fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.suptitle('Marginal Value of Each New Group of Compositions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comp_learning_curve_marginal.png', dpi=150, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(nc, ntr, c=r2_test, cmap='RdYlGn', s=80, edgecolors='k', lw=0.4, zorder=5)
plt.colorbar(sc, ax=ax, label='R² (test)')
for x_, y_, r2_ in zip(nc, ntr, r2_test):
    ax.annotate(f'R²={r2_:.3f}', (x_, y_), xytext=(4, 4), textcoords='offset points', fontsize=7)
ax.set_xlabel('Number of Compositions in Training', fontsize=12)
ax.set_ylabel('Number of Rows in Training', fontsize=12)
ax.set_title('Composition Count vs Row Count — coloured by Test R²', fontsize=12)
ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comp_vs_rows_scatter.png', dpi=150, bbox_inches='tight'); plt.show()

last3_dr2  = float(np.mean(np.abs(d_r2[-3:])))  if len(d_r2)  >= 3 else float('nan')
last3_dmae = float(np.mean(np.abs(d_mae[-3:]))) if len(d_mae) >= 3 else float('nan')
print('\n' + '=' * 70)
print('DATA SUFFICIENCY VERDICT')
print('=' * 70)
print(f'  Mean |ΔR²|  over last 3 steps : {last3_dr2:.6f}')
print(f'  Mean |ΔMAE| over last 3 steps : {last3_dmae:.6f}')
if last3_dr2 < 0.005 and last3_dmae < 0.005:
    print('  >> PLATEAU REACHED')
else:
    print('  >> CURVE STILL IMPROVING')
print('=' * 70)


## Per-Composition SHAP Analysis (TreeExplainer)

In [ ]:
print('=' * 70)
print('PER-COMPOSITION SHAP ANALYSIS')
print('=' * 70)

try:
    import shap
    print(f'  shap version: {shap.__version__}')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

os.makedirs(f'{RESULTS_DIR}/shap_per_composition', exist_ok=True)

print('\nBuilding SHAP TreeExplainer on full-data model ...')
explainer = shap.TreeExplainer(full_lc_model)

print('Computing SHAP values for all training rows ...')
shap_values_all = explainer.shap_values(full_train_X)
print(f'  SHAP matrix shape: {shap_values_all.shape}')

unique_comps = sorted(set(full_train_alloys))
print(f'\nCompositions to analyse: {len(unique_comps)}')
for c in unique_comps:
    n = (full_train_alloys == c).sum()
    print(f'  {c:40s} — {n} rows')

shap_summary = {}

for comp in unique_comps:
    mask     = full_train_alloys == comp
    sv_comp  = shap_values_all[mask]
    x_comp   = full_train_X[mask]
    mean_abs = np.abs(sv_comp).mean(axis=0)
    shap_summary[comp] = mean_abs
    n_rows = mask.sum()
    print(f'\n  Composition: {comp}  ({n_rows} rows)')

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    ax = axes[0]
    shap.summary_plot(sv_comp, x_comp, feature_names=feat_names, show=False, plot_size=None)
    plt.sca(ax)
    plt.title(f'{comp}  |  SHAP Summary (n={n_rows})', fontsize=11, fontweight='bold')

    ax = axes[1]
    order  = np.argsort(mean_abs)[::-1]
    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(feat_names)))
    ax.barh([feat_names[i] for i in order], mean_abs[order], color=colors, edgecolor='k', linewidth=0.4)
    ax.set_xlabel('Mean |SHAP value|', fontsize=11)
    ax.set_title(f'{comp}  |  Feature Importance (mean |SHAP|)', fontsize=11)
    ax.grid(axis='x', linestyle='--', alpha=0.4)

    plt.tight_layout()
    safe_name = comp.replace('/', '_').replace(' ', '_')
    fig_path  = f'{RESULTS_DIR}/shap_per_composition/shap_{safe_name}.png'
    plt.savefig(fig_path, dpi=130, bbox_inches='tight')
    plt.close()
    print(f'    Saved -> {fig_path}')

print('\n[Per-composition SHAP plots saved]')


In [ ]:
print('=' * 70)
print('SHAP COMPOSITE COMPARISON FIGURES')
print('=' * 70)

n_comp_  = len(unique_comps)
shap_mat = np.zeros((n_comp_, len(feat_names)))
for i, comp in enumerate(unique_comps):
    shap_mat[i] = shap_summary[comp]

shap_mat_norm = shap_mat / (shap_mat.sum(axis=1, keepdims=True) + 1e-12)

fig_h = max(6, n_comp_ * 0.45)
fig, ax = plt.subplots(figsize=(max(12, len(feat_names)*1.8), fig_h))
im = ax.imshow(shap_mat_norm, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Normalised mean |SHAP|')
ax.set_xticks(range(len(feat_names)))
ax.set_xticklabels(feat_names, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(n_comp_))
ax.set_yticklabels(unique_comps, fontsize=8)
for i in range(n_comp_):
    for j in range(len(feat_names)):
        ax.text(j, i, f'{shap_mat_norm[i, j]:.2f}', ha='center', va='center', fontsize=6.5,
                color='black' if shap_mat_norm[i, j] < 0.6 else 'white')
ax.set_title('Per-Composition SHAP Heatmap\n(normalised mean |SHAP| — brighter = more important for that alloy)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
hm_path = f'{RESULTS_DIR}/shap_composition_heatmap.png'
plt.savefig(hm_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved -> {hm_path}')

top_feat_idx   = shap_mat.argmax(axis=1)
top_feat_vals  = shap_mat.max(axis=1)
top_feat_names = [feat_names[i] for i in top_feat_idx]
feat_color_map = {f: plt.cm.tab10(k / len(feat_names)) for k, f in enumerate(feat_names)}
bar_colors = [feat_color_map[f] for f in top_feat_names]

fig, ax = plt.subplots(figsize=(max(12, n_comp_ * 0.7), 6))
bars = ax.bar(unique_comps, top_feat_vals, color=bar_colors, edgecolor='k', linewidth=0.5)
ax.set_xlabel('Composition', fontsize=11)
ax.set_ylabel('Mean |SHAP| of dominant feature', fontsize=11)
ax.set_title('Most Important Feature per Composition (SHAP)', fontsize=12, fontweight='bold')
ax.set_xticklabels(unique_comps, rotation=55, ha='right', fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.4)
for bar, fname in zip(bars, top_feat_names):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + top_feat_vals.max() * 0.01,
            fname, ha='center', va='bottom', fontsize=6.5, rotation=45)
handles = [plt.Rectangle((0,0),1,1, color=feat_color_map[f]) for f in feat_names]
ax.legend(handles, feat_names, title='Feature', fontsize=7, loc='upper right', ncol=2)
plt.tight_layout()
bar_path = f'{RESULTS_DIR}/shap_top_feature_per_comp.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved -> {bar_path}')

global_shap = shap_mat.mean(axis=0)
order_g   = np.argsort(global_shap)[::-1]
colors_g  = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(feat_names)))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([feat_names[i] for i in order_g], global_shap[order_g], color=colors_g, edgecolor='k', linewidth=0.5)
ax.set_ylabel('Global Mean |SHAP|  (averaged across all compositions)', fontsize=11)
ax.set_xlabel('Feature', fontsize=11)
ax.set_title('Global Feature Importance via SHAP\n(pooled across all compositions)', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=40)
ax.grid(axis='y', linestyle='--', alpha=0.4)
for i, (f_i, v) in enumerate(zip(order_g, global_shap[order_g])):
    ax.text(i, v + global_shap.max() * 0.01, f'{v:.3f}', ha='center', fontsize=8)
plt.tight_layout()
glob_path = f'{RESULTS_DIR}/shap_global_bar.png'
plt.savefig(glob_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved -> {glob_path}')

fig, ax = plt.subplots(figsize=(max(14, n_comp_ * 0.8), 7))
x_pos  = np.arange(n_comp_)
bottom = np.zeros(n_comp_)
feat_colors_list = [plt.cm.tab10(k / len(feat_names)) for k in range(len(feat_names))]
for j, feat in enumerate(feat_names):
    vals = shap_mat[:, j]
    ax.bar(x_pos, vals, bottom=bottom, label=feat, color=feat_colors_list[j], edgecolor='k', linewidth=0.3)
    bottom += vals
ax.set_xticks(x_pos)
ax.set_xticklabels(unique_comps, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Mean |SHAP| (stacked)', fontsize=11)
ax.set_title('Stacked SHAP Feature Contributions per Composition', fontsize=12, fontweight='bold')
ax.legend(title='Feature', fontsize=8, loc='upper right', ncol=2, bbox_to_anchor=(1.14, 1))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
stack_path = f'{RESULTS_DIR}/shap_stacked_per_comp.png'
plt.savefig(stack_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved -> {stack_path}')

shap_csv = pd.DataFrame(shap_mat, columns=feat_names, index=pd.Index(unique_comps, name='composition'))
shap_csv['dominant_feature'] = top_feat_names
shap_csv.to_csv(f'{RESULTS_DIR}/shap_per_composition_summary.csv')
print(f'Saved -> {RESULTS_DIR}/shap_per_composition_summary.csv')
print('\n' + '=' * 70)
print('SHAP ANALYSIS COMPLETE')
print('=' * 70)
